# 面试问题：GPTQ 与 AWQ 的核心区别是什么？4-bit weight-only quantization 怎样从零实现和评估？

**一句话回答**：普通 RTN 只最小化权重舍入误差；GPTQ 用校准激活形成近似 Hessian，逐列量化并把误差补偿到未量化权重；AWQ 用激活统计识别重要输入通道，通过等价缩放降低其量化误差。二者都是 data-aware PTQ，但优化目标和算法不同；是否加速还取决于真实 INT4 packing/kernel。

本 Notebook 实现 groupwise symmetric RTN、输出误差/Hessian 二次型、单步二阶补偿、AWQ 风格缩放搜索、INT4 nibble packing、dequant GEMM、分组/outlier 分析和发布门禁，并用中文注释标明教学近似与生产算法边界。


In [ ]:
import math  # 导入本单元所需的依赖。
import numpy as np  # 导入本单元所需的依赖。

# 校准激活与权重使用固定随机种子，便于误差回归。
SEED144=14401; rng144=np.random.default_rng(SEED144)  # 计算并保存当前步骤的中间状态。
assert SEED144==14401  # 用受控断言验证关键不变量。
assert 2**4==16  # 用受控断言验证关键不变量。
assert np.isfinite(rng144.normal())  # 用受控断言验证关键不变量。


## 1. 基线是 groupwise symmetric round-to-nearest

每组 scale=`max|w|/qmax`，int4 有符号范围常用 `[-7,7]` 或具体 kernel 约定。group 越小 scale 越贴合局部分布，但 metadata 与 kernel 复杂度增加。零 scale 组必须特殊处理，避免除零。


In [ ]:
def quant_group144(x,bits=4):  # 定义本节可复用的核心函数。
    # 对单组做对称量化并立即反量化，返回整数与 scale。
    x=np.asarray(x,dtype=float); qmax=2**(bits-1)-1; scale=max(float(np.max(np.abs(x)))/qmax,1e-12); q=np.clip(np.rint(x/scale),-qmax,qmax).astype(np.int8)  # 计算并保存当前步骤的中间状态。
    return q,scale,q.astype(float)*scale  # 返回当前分支计算出的结果。
vec144=np.array([-1.,-.2,0.,.3,.9]); q144,s144,dq144=quant_group144(vec144)  # 计算并保存当前步骤的中间状态。
assert q144.min()>=-7 and q144.max()<=7  # 用受控断言验证关键不变量。
assert s144>0  # 用受控断言验证关键不变量。
assert np.max(np.abs(vec144-dq144))<=s144/2+1e-12  # 用受控断言验证关键不变量。


## 2. LLM PTQ 关心层输出误差，不只是权重 MSE

对校准输入 `X`，量化层误差是 `||X(W-Wq)||²`。高激活通道上的小权重误差可能比低激活通道的大误差更重要。下面按整矩阵一个 scale 建立 RTN baseline，并同时记录 weight MSE 与 output MSE。


In [ ]:
X144=rng144.normal(size=(256,6)); X144[:,0]*=8; W144=rng144.normal(size=(6,4))  # 计算并保存当前步骤的中间状态。
def quant_matrix144(W):  # 定义本节可复用的核心函数。
    # 教学基线使用全矩阵单 scale；后续再讨论 group 粒度。
    q,s,dq=quant_group144(W.ravel()); return q.reshape(W.shape),s,dq.reshape(W.shape)  # 计算并保存当前步骤的中间状态。
Qbase144,Sbase144,Wbase144=quant_matrix144(W144); weight_mse144=float(np.mean((W144-Wbase144)**2)); output_mse144=float(np.mean((X144@W144-X144@Wbase144)**2))  # 计算并保存当前步骤的中间状态。
assert weight_mse144>=0 and output_mse144>=0  # 用受控断言验证关键不变量。
assert Qbase144.shape==W144.shape  # 用受控断言验证关键不变量。
assert Sbase144>0  # 用受控断言验证关键不变量。


## 3. 校准 Hessian 近似把输出误差写成二次型

线性层下 `H=XᵀX/n`，权重误差 `E=Wq-W` 的平均输出平方误差等于 `trace(EᵀHE)`。GPTQ 使用该二阶结构和逆 Hessian 做顺序误差补偿；实际算法还包含阻尼、列顺序、block 更新和数值稳定处理。


In [ ]:
H144=X144.T@X144/len(X144)  # 计算并保存当前步骤的中间状态。
def hessian_error144(W,Wq,H):  # 定义本节可复用的核心函数。
    # trace(E^T H E) 与校准 batch 的线性层输出 MSE 总和对应。
    E=Wq-W; return float(np.trace(E.T@H@E))  # 计算并保存当前步骤的中间状态。
h_err144=hessian_error144(W144,Wbase144,H144)  # 计算并保存当前步骤的中间状态。
assert np.allclose(H144,H144.T)  # 用受控断言验证关键不变量。
assert np.linalg.eigvalsh(H144).min()>=-1e-10  # 用受控断言验证关键不变量。
assert math.isclose(h_err144,float(np.mean(np.sum((X144@W144-X144@Wbase144)**2,axis=1))),rel_tol=1e-10)  # 用受控断言验证关键不变量。


## 4. 二阶补偿把已量化列的误差传播到剩余列

固定某输入通道的舍入误差 `d_i` 后，令剩余误差 `d_R=-H_RR^{-1}H_Ri d_i` 可降低局部二次型；完整 GPTQ 会逐列重复类似更新。下面只演示单输出、单步最优补偿，明确它不是完整高性能 GPTQ 实现。


In [ ]:
i144=0; rest144=np.arange(1,H144.shape[0]); d_i144=.1  # 计算并保存当前步骤的中间状态。
# 解局部二次问题，计算未量化通道应吸收的误差。
d_plain144=np.zeros(6); d_plain144[i144]=d_i144; d_comp144=d_plain144.copy(); d_comp144[rest144]=-np.linalg.solve(H144[np.ix_(rest144,rest144)],H144[rest144,i144]*d_i144)  # 计算并保存当前步骤的中间状态。
cost_plain144=float(d_plain144@H144@d_plain144); cost_comp144=float(d_comp144@H144@d_comp144)  # 计算并保存当前步骤的中间状态。
assert cost_comp144<=cost_plain144+1e-12  # 用受控断言验证关键不变量。
assert d_comp144[0]==d_i144  # 用受控断言验证关键不变量。
assert np.any(np.abs(d_comp144[1:])>0)  # 用受控断言验证关键不变量。


## 5. AWQ 风格等价缩放用 activation salience 保护通道

对输入通道 scale `s`，`XW=(X/s)(sW)` 完全等价；先放大重要通道权重再量化，可能降低输出误差。AWQ 会搜索缩放强度并配合硬件友好实现。这里用 `mean|X|^α` 扫 α，选择校准输出误差最小者。


In [ ]:
sal144=np.mean(np.abs(X144),axis=0)+1e-6  # 计算并保存当前步骤的中间状态。
def awq_candidate144(alpha):  # 定义本节可复用的核心函数。
    # 缩放输入与权重方向相反，浮点层输出保持严格不变。
    s=(sal144/sal144.mean())**alpha; Xs=X144/s; Ws=W144*s[:,None]; _,_,Wqs=quant_matrix144(Ws); return s,float(np.mean((X144@W144-Xs@Wqs)**2)),Xs,Wqs  # 计算并保存当前步骤的中间状态。
candidates144=[awq_candidate144(a) for a in np.linspace(0,1,11)]; best_awq144=min(candidates144,key=lambda x:x[1])  # 计算并保存当前步骤的中间状态。
assert best_awq144[1]<=output_mse144+1e-12  # 用受控断言验证关键不变量。
assert np.allclose((X144/best_awq144[0])@(W144*best_awq144[0][:,None]),X144@W144)  # 用受控断言验证关键不变量。
assert np.all(best_awq144[0]>0)  # 用受控断言验证关键不变量。


## 6. 真正省内存需要把两个 4-bit 值打进一个 byte

只把 int4 存在 int8 数组并没有达到 4 bit/weight。下面把 `[0,15]` 无符号码的低/高 nibble 打包，奇数长度补零并保存逻辑长度。生产 kernel 还要求特定 tile、对齐、scale layout 和 fused dequant。


In [ ]:
def pack_u4_144(q):  # 定义本节可复用的核心函数。
    # 两个 4-bit 码分别放入一个字节的低四位和高四位。
    q=np.asarray(q,dtype=np.uint8); n=len(q); q=np.pad(q,(0,n%2)); return (q[0::2]|(q[1::2]<<4)).astype(np.uint8),n  # 计算并保存当前步骤的中间状态。
def unpack_u4_144(packed,n):  # 定义本节可复用的核心函数。
    out=np.empty(len(packed)*2,dtype=np.uint8); out[0::2]=packed&15; out[1::2]=(packed>>4)&15; return out[:n]  # 计算并保存当前步骤的中间状态。
codes144=np.array([0,1,7,15,9],dtype=np.uint8); packed144,ncodes144=pack_u4_144(codes144)  # 计算并保存当前步骤的中间状态。
assert np.array_equal(unpack_u4_144(packed144,ncodes144),codes144)  # 用受控断言验证关键不变量。
assert len(packed144)==math.ceil(len(codes144)/2)  # 用受控断言验证关键不变量。
assert packed144.dtype==np.uint8  # 用受控断言验证关键不变量。


## 7. Group size、outlier 与 calibration domain 共同决定误差

小 group 减少不同幅度权重共享 scale 的冲突；保留少量 outlier 高精度或做 AWQ scaling 也可改善。但 metadata、反量化和稀疏分支可能抵消速度。校准集若不覆盖代码/长上下文等部署域，data-aware 方法会过拟合错误分布。


In [ ]:
def group_quant144(v,group):  # 定义本节可复用的核心函数。
    # 每段独立量化，返回与原向量等长的反量化结果。
    out=[]  # 计算并保存当前步骤的中间状态。
    for start in range(0,len(v),group): out.extend(quant_group144(v[start:start+group])[2])  # 遍历输入元素以累积或检查结果。
    return np.asarray(out)  # 返回当前分支计算出的结果。
outlier_vec144=np.array([.1,.2,.15,8.,.1,.2,.15,.3]); coarse144=group_quant144(outlier_vec144,8); fine144=group_quant144(outlier_vec144,4)  # 计算并保存当前步骤的中间状态。
assert np.mean((fine144-outlier_vec144)**2)<=np.mean((coarse144-outlier_vec144)**2)+1e-12  # 用受控断言验证关键不变量。
assert len(fine144)==len(outlier_vec144)  # 用受控断言验证关键不变量。
assert np.all(np.isfinite(fine144))  # 用受控断言验证关键不变量。


## 8. 模型更小不自动等于端到端更快

发布需比较 weight/output reconstruction、PPL、任务/安全 slice、不同层敏感度；系统侧测真实 packed bytes、峰值 HBM、prefill/decode tokens/s、TTFT/TPOT 与 kernel 覆盖率。若运行时先完整反量化到 FP16，容量可能省但每步速度未必提升。


In [ ]:
params144=7_000_000_000; fp16_gb144=params144*2/1e9; int4_gb144=params144*.5/1e9  # 计算并保存当前步骤的中间状态。
report144={"fp16_gb":fp16_gb144,"int4_weight_gb":int4_gb144,"compression":fp16_gb144/int4_gb144,"calib_output_mse":best_awq144[1]}  # 计算并保存当前步骤的中间状态。
# 这里只证明理论权重字节，速度仍需真实 kernel benchmark。
assert report144["compression"]==4  # 用受控断言验证关键不变量。
assert report144["int4_weight_gb"]==3.5  # 用受控断言验证关键不变量。
assert report144["calib_output_mse"]>=0  # 用受控断言验证关键不变量。


## 面试总结

完整回答是：**RTN/group scale baseline → 用 `X(W-Wq)` 看输出误差 → `H=XᵀX/n` 建二阶目标 → GPTQ 类逐列量化/逆 Hessian 补偿 → AWQ 类 activation salience 等价缩放 → INT4 真正 nibble packing → 扫 group/outlier/calibration → PPL/任务/安全回归 → packed HBM 与真实 kernel 的 prefill/decode benchmark**。GPTQ 与 AWQ 都是校准感知 PTQ，但不能混称同一个算法。

延伸阅读：[GPTQ](https://arxiv.org/abs/2210.17323)、[AWQ](https://arxiv.org/abs/2306.00978)、[SmoothQuant](https://arxiv.org/abs/2211.10438)。
